# BDB27 — full-data game turns, counted on your laptop

**This notebook is the whole interface.** Put the folder holding your files in `DATA_DIR` (§1)
and press **Run All**. No terminal, no commands to type.

**What it produces.** The same turn table stage 02 builds, from the *full* `game_tracking.csv`:
how many turns each position group actually makes in games, in figure 1's six bands — 30–60°,
60–90°, 90–120°, 120–150°, 150–180° and >180°. The detector stores 30–45° and 45–60° separately;
the 30–60° row is formed **only when reporting**, exactly as figure 1 does it.

**Why it is trustworthy.** No detection maths lives in this notebook. It imports the pipeline's
own `Engine`, and reads every threshold from the repo's `config.yaml` (`turn_coverage:` block).
Before it touches full data it re-runs the pipeline on the repo's 1,048,575-row 2023 sample and
asserts the four published numbers plus a row-for-row match against
`outputs/figures/fig_the_hole.csv`. A mismatch **stops the notebook** with the exact diff.

**It never loads the tracking file into memory.** DuckDB converts the CSV to Parquet once, then
reads it back a group of games at a time (`resources.game_chunk_games`, 12 by default). Every
chunk is checkpointed, so closing the laptop and re-running resumes where it stopped.

## 1. Paths and mode — the only cell to edit

* `DATA_DIR` — the **folder** holding your `game_tracking.csv`, `player_play.csv` and
  `players.csv`. `None` means the repo's own `data/raw`.
* If `combine_tracking.csv` and `combine_results.csv` sit in the same folder they are used too
  (they are cheap), so the Combine side of the 30–60° comparison is reported beside the game
  side. Their absence is never an error — the notebook says which side it could not compute.
* `INPUT_MODE` — `"full"` analyses `game_tracking.csv`; `"sample"` analyses the repo's 2023
  sample, which is what a dry run on this machine uses. The **fidelity gate always runs on the
  sample**, whatever this is set to.

Everything below §1 is fixed. Do not edit it.

In [ ]:
# ======================= CONFIG — the only cell to edit =========================
DATA_DIR   = None       # folder with game_tracking.csv / player_play.csv / players.csv
INPUT_MODE = "full"     # "full" = your game_tracking.csv | "sample" = repo 2023 sample
# =============================================================================

## 2. Setup

Finds the repo (walks up until it sees `config.yaml` and `src/bdb27`), puts `src/` on the path,
and checks the libraries the pipeline needs. A package is installed **only** if it is genuinely
missing; an installed environment is left alone.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

_NEEDED = {"duckdb": "duckdb", "pandas": "pandas", "pyarrow": "pyarrow",
           "numpy": "numpy", "yaml": "pyyaml"}
_missing = [pip for mod, pip in _NEEDED.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print(f"Installing missing package(s): {', '.join(_missing)} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *_missing])
else:
    print("All required packages are already installed — nothing installed.")


def find_repo_root() -> Path:
    '''Walk upward from the notebook's working directory.'''
    here = Path.cwd().resolve()
    for parent in (here, *here.parents):
        if (parent / "config.yaml").exists() and (parent / "src" / "bdb27").is_dir():
            return parent
    raise RuntimeError(
        "Cannot locate the BDB27 repo root (need config.yaml + src/bdb27). "
        "Open this notebook from inside the BDB27 checkout.")


REPO = find_repo_root()
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

from bdb27 import common
from bdb27.turns import TURN_COLUMNS, TURN_SCHEMA, Engine, frame_of

cfg = common.load_config(REPO / "config.yaml")
print(f"repo         : {REPO}")
print(f"config_sha   : {common.config_fingerprint(cfg)}")
print(f"git rev      : {common.git_rev(REPO)}")
print(f"chunk size   : {int(cfg.get('resources.game_chunk_games'))} games")

## 3. Inputs

Resolves the data folder, names every file it will read, and prints a present/absent line for
each one. A missing required file is always named — nothing is silently substituted.

In [ ]:
import time

t_start = time.time()

#: Where the notebook writes everything. Already gitignored: nothing here is repo content.
RUN = REPO / "bdb27_run"
RUN.mkdir(exist_ok=True)
(RUN / "tmp").mkdir(exist_ok=True)

if DATA_DIR is None:
    DATA = REPO / "data" / "raw"
else:
    DATA = Path(DATA_DIR).expanduser().resolve()
    if not DATA.is_dir():
        raise RuntimeError(f"DATA_DIR does not exist or is not a folder: {DATA}")

#: The tracking file's name is the ONLY difference between the two modes.
if INPUT_MODE == "full":
    TRACKING_NAME = cfg.get("files.game_tracking")
elif INPUT_MODE == "sample":
    TRACKING_NAME = cfg.get("sample_files.game_tracking")
else:
    raise RuntimeError(f"INPUT_MODE must be 'full' or 'sample', got {INPUT_MODE!r}")

GAME_CSV = DATA / TRACKING_NAME
PP_CSV = DATA / cfg.get("files.player_play")
PLAYERS_CSV = DATA / cfg.get("files.players")
COMBINE_CSV = DATA / cfg.get("files.combine_tracking")
COMBINE_RESULTS_CSV = DATA / cfg.get("files.combine_results")
#: The repo's sample, used by the fidelity gate in §6 whatever INPUT_MODE says.
SAMPLE_CSV = REPO / "data" / "raw" / cfg.get("sample_files.game_tracking")
FIG_HOLE_CSV = REPO / "outputs" / "figures" / "fig_the_hole.csv"

print(f"data folder  : {DATA}")
for p in (GAME_CSV, PP_CSV, PLAYERS_CSV):
    size = f"{p.stat().st_size / 1e6:,.1f} MB" if p.exists() else "ABSENT"
    print(f"  {'ok ' if p.exists() else 'MISSING'} {p.name:<34} {size}")
for p in (COMBINE_CSV, COMBINE_RESULTS_CSV):
    size = f"{p.stat().st_size / 1e6:,.1f} MB" if p.exists() else "ABSENT"
    print(f"  {'ok ' if p.exists() else '---'} {p.name:<34} {size}  (optional)")
missing = [p.name for p in (GAME_CSV, PP_CSV, PLAYERS_CSV) if not p.exists()]
if missing:
    raise RuntimeError(f"Missing required input(s) in {DATA}: {', '.join(missing)}")

## 4. Convert once — CSV → Parquet with DuckDB

The tracking CSV is read once and written as Parquet, **sorted by
`game_id, play_id, nfl_id, time` and carrying only the columns the detector uses**
(`game_id, play_id, nfl_id, time, x, y, o, dir`). Sorting once is what makes every later read a
narrow per-game read instead of a scan of the whole file.

This step is checkpointed: if the Parquet exists and is newer than the CSV, conversion is skipped.

In [ ]:
import duckdb


def convert(csv_path: Path, columns, out_path: Path, order_by: str) -> Path:
    '''One CSV -> one Parquet, checkpointed. Returns the Parquet path.'''
    if out_path.exists() and out_path.stat().st_mtime >= csv_path.stat().st_mtime:
        print(f"  reuse   {out_path.name}")
        return out_path
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = out_path.with_suffix(".parquet.tmp")
    con = common.connect(cfg)                       # memory limit + threads from config
    con.execute(f"SET temp_directory='{RUN / 'tmp'}'")
    con.execute("SET preserve_insertion_order=false")
    sniff = int(cfg.get("resources.csv_sniff_rows"))
    t0 = time.time()
    con.execute(f'''
        COPY (
            SELECT {", ".join(columns)}
            FROM read_csv_auto('{csv_path}', header=true,
                               sample_size={sniff}, nullstr='NA')
            ORDER BY {order_by}
        ) TO '{tmp}' (FORMAT PARQUET, COMPRESSION ZSTD)
        ''')
    con.close()
    tmp.replace(out_path)
    print(f"  convert {out_path.name}  {time.time() - t0:,.1f}s")
    return out_path


#: The detector reads exactly these columns; only x/y/time (plus o, as a flag) are used.
TRACKING_COLUMNS = ["game_id", "play_id", "nfl_id", "time", "x", "y", "o", "dir"]
_undeclared = [c for c in TRACKING_COLUMNS if c not in cfg.get("columns.game_tracking")]
if _undeclared:
    raise RuntimeError(f"config.yaml columns.game_tracking is missing {_undeclared}")

t_convert = time.time()
print(f"converting {GAME_CSV.name} ...")
GAME_PQ = convert(GAME_CSV, TRACKING_COLUMNS, RUN / f"{GAME_CSV.stem}.parquet",
                  "game_id, play_id, nfl_id, time")
PP_PQ = convert(PP_CSV, cfg.get("columns.player_play"), RUN / "player_play.parquet",
                "game_id, play_id, nfl_id")
PLAYERS_PQ = convert(PLAYERS_CSV, ["nfl_id", "display_name", "draft_year", "nfl_position"],
                     RUN / "players.parquet", "nfl_id")
COMBINE_PQ = (convert(COMBINE_CSV, cfg.get("columns.combine_tracking"),
                      RUN / "combine_tracking.parquet",
                      "drill_name, event_id, nfl_id, attempt, time")
              if COMBINE_CSV.exists() else None)
COMBINE_RESULTS_PQ = (convert(COMBINE_RESULTS_CSV, cfg.get("columns.combine_results"),
                              RUN / "combine_results.parquet", "nfl_id")
                      if COMBINE_CSV.exists() and COMBINE_RESULTS_CSV.exists() else None)
print(f"conversion total {time.time() - t_convert:,.1f}s")

_con = common.connect(cfg)
n_rows, n_games, n_plays, n_players = _con.execute(f'''
    SELECT count(*), count(DISTINCT game_id),
           count(DISTINCT (game_id, play_id)), count(DISTINCT nfl_id)
    FROM read_parquet('{GAME_PQ}')''').fetchone()
_con.close()
print(f"\ntracking parquet : {n_rows:,} rows | {n_games:,} games | "
      f"{n_plays:,} plays | {n_players:,} players")

## 5. Detect — the pipeline's own engine

`Engine.from_config(cfg, "game")` is imported from `src/bdb27/turns.py`: the smoothing, heading,
angular rate and turn segmentation are the ones stage 02 uses, with every threshold read from
`config.yaml`. Nothing is re-implemented here. This cell only builds the same context dictionary
stage 02 builds (`position_group` from `positions.game_families`, `alignment`/`snap_type` from
`player_play`, name and draft year from `players`) and feeds frames to the engine in game chunks.

In [ ]:
import json

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

ENGINE = Engine.from_config(cfg, "game")
POSITION_FAMILIES = cfg.get("positions.game_families")


def position_family(nfl_position) -> str:
    '''Map an NFL position to its reporting family (config, single source of truth).'''
    for group, members in POSITION_FAMILIES.items():
        if nfl_position in members:
            return group
    return "OTHER"


def snap_type_sql(col: str = "") -> str:
    '''The snap-type CASE from config's snap_types.rules (the rule set stage 02 uses).'''
    parts = []
    for rule in cfg.get("snap_types.rules"):
        test, name, column = rule["test"], rule["name"], rule.get("column")
        if test == "notnull":
            cond = f"{col}{column} IS NOT NULL"
        elif test == "truthy":
            cond = f"{col}{column}"
        elif test == "notnull_excluding":
            vals = ", ".join("'" + str(v) + "'" for v in rule["exclude"])
            cond = f"({col}{column} IS NOT NULL AND {col}{column} NOT IN ({vals}))"
        else:
            cond = "TRUE"
        parts.append(f"WHEN {cond} THEN '{name}'")
    return "CASE " + " ".join(parts) + " ELSE 'OTHER' END"


def load_players(con) -> dict:
    df = con.execute(
        f"SELECT nfl_id, display_name, draft_year, nfl_position FROM read_parquet('{PLAYERS_PQ}')"
    ).df()
    return df.set_index("nfl_id").to_dict("index")


def draft_year_of(players: dict, nfl_id: int):
    '''Draft year, or None — pandas gives NaN for a missing year, which is not an int.'''
    value = players.get(nfl_id, {}).get("draft_year")
    return int(value) if value is not None and pd.notna(value) else None


def arrow_table(frame: pd.DataFrame) -> pa.Table:
    '''Cast a turn frame to the pipeline's own TURN_SCHEMA (same dtypes as stage 02).'''
    arrays = [common._arrow_array(frame[f.name], f.type) for f in TURN_SCHEMA]
    return pa.Table.from_arrays(arrays, schema=TURN_SCHEMA)


def write_part(rows: list, part: Path) -> int:
    tmp = part.with_suffix(".parquet.tmp")
    frame = frame_of(rows, TURN_SCHEMA)
    pq.write_table(arrow_table(frame), tmp, compression="zstd")
    tmp.replace(part)
    return len(frame)


def detect_game_chunks(outdir: Path, tracking_pq: Path, label: str) -> list:
    '''Run the engine over every (game, play, player), one group of games at a time.

    Writes one checkpoint Parquet per chunk plus a small JSON receipt, so a re-run
    skips completed chunks. Returns the per-chunk receipts.
    '''
    outdir.mkdir(parents=True, exist_ok=True)
    con = common.connect(cfg)
    games = [int(g) for g in con.execute(
        f"SELECT DISTINCT game_id FROM read_parquet('{tracking_pq}') ORDER BY 1"
    ).fetchnumpy()["game_id"]]
    size = int(cfg.get("resources.game_chunk_games"))
    players = load_players(con)
    sha = common.config_fingerprint(cfg)

    receipts, t0, done_rows = [], time.time(), 0
    n_chunks = (len(games) + size - 1) // size
    print(f"{label}: {len(games):,} games in {n_chunks} chunk(s) of {size}")
    for k, start in enumerate(range(0, len(games), size)):
        batch = games[start:start + size]
        part = outdir / f"chunk_{k:05d}.parquet"
        receipt_path = outdir / f"chunk_{k:05d}.json"
        if part.exists() and receipt_path.exists():
            old = json.loads(receipt_path.read_text())
            if old.get("config_sha") == sha and old.get("n_games") == len(batch):
                receipts.append(old)
                done_rows += int(old["rows"])
                print(f"  [{k + 1}/{n_chunks}] reused  chunk {k:05d} ({old['rows']:,} turns)")
                continue
        t_chunk = time.time()
        ph = ",".join("?" * len(batch))
        df = con.execute(f'''
            SELECT game_id, play_id, nfl_id, epoch_ms(time) AS ms, x, y, o, dir
            FROM read_parquet(?) WHERE game_id IN ({ph})''', [str(tracking_pq), *batch]).df()
        df = df.sort_values(["game_id", "play_id", "nfl_id", "ms"], kind="stable")
        pp = con.execute(f'''
            SELECT game_id, play_id, nfl_id, lined_up_position,
                   {snap_type_sql()} AS snap_type
            FROM read_parquet(?) WHERE game_id IN ({ph})''', [str(PP_PQ), *batch]).df()
        pp_idx = pp.set_index(["game_id", "play_id", "nfl_id"]).to_dict("index")
        rows, n_snaps = [], 0
        for (gid, pid, nid), g in df.groupby(["game_id", "play_id", "nfl_id"],
                                             observed=True, sort=True):
            info = pp_idx.get((gid, pid, nid), {})
            nid = int(nid)
            npos = players.get(nid, {}).get("nfl_position")
            ctx = {
                "drill_name": None, "event_id": None, "attempt": None,
                "game_id": int(gid), "play_id": int(pid), "nfl_id": nid,
                "player_name": players.get(nid, {}).get("display_name"),
                "position": npos, "position_group": position_family(npos),
                "alignment": info.get("lined_up_position"),
                "snap_type": info.get("snap_type"),
                "draft_year": draft_year_of(players, nid),
            }
            _att, trs = ENGINE.analyse(g.ms.to_numpy(), g.x.to_numpy(), g.y.to_numpy(), ctx,
                                       o=g.o.to_numpy(), dirv=g.dir.to_numpy())
            rows.extend(trs)
            n_snaps += 1
        rows_written = write_part(rows, part)
        dt = time.time() - t_chunk
        receipt = {
            "chunk": k, "label": label, "games": [int(batch[0]), int(batch[-1])],
            "n_games": len(batch), "snaps": n_snaps, "rows": rows_written,
            "frames": int(len(df)), "seconds": round(dt, 2), "config_sha": sha,
        }
        receipt_path.write_text(json.dumps(receipt, indent=2))
        receipts.append(receipt)
        done_rows += rows_written
        elapsed = time.time() - t0
        eta = elapsed / (k + 1) * (n_chunks - k - 1)
        print(f"  [{k + 1}/{n_chunks}] games {batch[0]}..{batch[-1]}  "
              f"{len(df):,} rows -> {rows_written:,} turns  {dt:,.1f}s  "
              f"| elapsed {elapsed / 60:,.1f} min | ETA {eta / 60:,.1f} min")
    con.close()
    print(f"{label}: {done_rows:,} turn rows in {n_chunks} chunk(s)")
    return receipts

### Combine side (optional — used only when both its files are present)

The same engine with `setting="combine"`, fed from `combine_tracking.csv`. It supplies the
Combine's own 30–60° share, which is the number the game share is read against.

In [ ]:
def detect_combine() -> list:
    '''Run the engine over the Combine attempts (one context row per attempt).'''
    if COMBINE_PQ is None or COMBINE_RESULTS_PQ is None:
        print("combine: skipped (combine_tracking.csv / combine_results.csv not present)")
        return []
    con = common.connect(cfg)
    engine = Engine.from_config(cfg, "combine")
    players = load_players(con)
    pos_map = con.execute(
        f"SELECT nfl_id, combine_position FROM read_parquet('{COMBINE_RESULTS_PQ}')"
    ).df().set_index("nfl_id").combine_position.to_dict()
    df = con.execute(f'''
        SELECT drill_name, event_id, nfl_id, attempt, epoch_ms(time) AS ms, x, y
        FROM read_parquet('{COMBINE_PQ}')
        WHERE entity_type = 'PLAYER'
        ORDER BY drill_name, event_id, nfl_id, attempt, ms''').df()
    dup = df.duplicated(["nfl_id", "event_id", "attempt", "ms"])
    dup_by_key = (df[dup].groupby(["event_id", "nfl_id", "attempt"]).size().to_dict()
                  if bool(dup.any()) else {})
    if cfg.get("sampling.drop_duplicate_frames"):
        df = df[~dup]
    rows, t0 = [], time.time()
    for (drill, event_id, nfl_id, attempt), g in df.groupby(
            ["drill_name", "event_id", "nfl_id", "attempt"], observed=True, sort=False):
        nid = int(nfl_id)
        ctx = {
            "drill_name": drill, "event_id": str(event_id), "attempt": int(attempt),
            "game_id": None, "play_id": None, "nfl_id": nid,
            "player_name": players.get(nid, {}).get("display_name"),
            "position": pos_map.get(nid), "position_group": pos_map.get(nid),
            "alignment": None, "snap_type": None,
            "draft_year": draft_year_of(players, nid),
        }
        _att, trs = engine.analyse(
            g.ms.to_numpy(), g.x.to_numpy(), g.y.to_numpy(), ctx,
            duplicate_frames=int(dup_by_key.get((event_id, nfl_id, attempt), 0)))
        rows.extend(trs)
    con.close()
    part = RUN / "combine_turns.parquet"
    n = write_part(rows, part)
    print(f"combine: {n:,} turn rows from {len(df):,} frames in {time.time() - t0:,.1f}s")
    return rows

## 6. Fidelity gate — must pass before any full-data work

The gate re-runs §4–§5 on the repo's 2023 sample (1,048,575 rows — the vendor's 2^20 − 1 row cap
— 333 games, 4,228 plays, 154 players) and requires **all** of:

* **2,262** valid DB game turns,
* **326** of them in the merged 30–60° band,
* **33** distinct DB players,
* **299** DB player-games,
* a row-for-row match against `outputs/figures/fig_the_hole.csv` — count, `setting_total` and
  `share_within_setting`, band for band, both settings when the Combine files are present.

Any mismatch prints the exact diff and **stops the notebook**: no full-data number is ever
produced past a failed gate. The gate has its own checkpoint folder, so its second run is seconds.

In [ ]:
EXPECTED = {
    "valid DB game turns": 2262,
    "DB game turns, 30-60 deg (merged)": 326,
    "DB game players": 33,
    "DB player-games": 299,
}
BAND_EDGES = [float(x) for x in cfg.get("figures.hole.bands.turn_angle_deg")]
BAND_OVERFLOW = cfg.get("figures.hole.bands.overflow_label")
BAND_LABELS = ([f"{lo:g}–{hi:g}°" for lo, hi in zip(BAND_EDGES[:-1], BAND_EDGES[1:])]
               + [f"{BAND_OVERFLOW}°"])
GATE_READY = False


def band_of(angle: pd.Series) -> pd.Series:
    '''Left-closed band labels from config; the last band is everything at/above the top edge.'''
    labels = pd.Series([None] * len(angle), index=angle.index, dtype=object)
    for (lo, hi), name in zip(zip(BAND_EDGES[:-1], BAND_EDGES[1:]), BAND_LABELS[:-1]):
        labels[(angle >= lo) & (angle < hi)] = name
    labels[angle >= BAND_EDGES[-1]] = BAND_LABELS[-1]
    return labels


def bands_table(turns: pd.DataFrame) -> pd.DataFrame:
    '''position_group x band x setting over VALID turns. Shares are within the setting.'''
    valid = turns[turns["valid"].fillna(False).astype(bool)].copy()
    valid["band"] = band_of(pd.to_numeric(valid["abs_turn_angle"], errors="coerce"))
    if valid["band"].isna().any():
        raise RuntimeError("some valid turns fall outside the configured angle bands")
    totals = (valid.groupby(["position_group", "setting"]).size()
              .rename("setting_total").reset_index())
    counts = (valid.groupby(["position_group", "band", "setting"]).size()
              .rename("count").reset_index())
    out = counts.merge(totals, on=["position_group", "setting"], how="left")
    out["band_index"] = out["band"].map({b: i + 1 for i, b in enumerate(BAND_LABELS)})
    out["band_low_deg"] = [BAND_EDGES[int(i) - 1] for i in out["band_index"]]
    out["band_high_deg"] = [BAND_EDGES[int(i)] if int(i) < len(BAND_EDGES) else None
                            for i in out["band_index"]]
    out["share_within_setting"] = out["count"] / out["setting_total"]
    cols = ["setting", "position_group", "band", "band_index", "band_low_deg",
            "band_high_deg", "count", "setting_total", "share_within_setting"]
    return (out[cols].sort_values(["setting", "position_group", "band_index"], kind="stable")
            .reset_index(drop=True))


REPORT_BANDS = ["30–60°"] + BAND_LABELS[2:]
REPORT_EDGES = [BAND_EDGES[0]] + BAND_EDGES[2:]


def merge_30_60(table: pd.DataFrame) -> pd.DataFrame:
    '''Reporting frame: 30–45° and 45–60° summed into 30–60°, every other band as stored.

    Called only when reporting — §6's comparison and §8's artefact. The stored table keeps
    30–45° and 45–60° separate; it is never merged.
    '''
    low = table[table["band"].isin(BAND_LABELS[:2])]
    merged = (low.groupby(["position_group", "setting"], as_index=False)
                 .agg(count=("count", "sum"), setting_total=("setting_total", "first")))
    merged["band"] = REPORT_BANDS[0]
    out = pd.concat([merged, table[table["band"].isin(BAND_LABELS[2:])]], ignore_index=True)
    out["share_within_setting"] = out["count"] / out["setting_total"]
    out["band_index"] = out["band"].map({b: i + 1 for i, b in enumerate(REPORT_BANDS)})
    out["band_low_deg"] = [REPORT_EDGES[int(i) - 1] for i in out["band_index"]]
    out["band_high_deg"] = [REPORT_EDGES[int(i)] if int(i) < len(REPORT_EDGES) else None
                            for i in out["band_index"]]
    cols = ["setting", "position_group", "band", "band_index", "band_low_deg",
            "band_high_deg", "count", "setting_total", "share_within_setting"]
    return (out[cols].sort_values(["setting", "position_group", "band_index"], kind="stable")
            .reset_index(drop=True))


gate_turns, table = None, None
if not SAMPLE_CSV.exists():
    print("GATE SKIPPED — the repo sample is not present on this machine.")
    print(f"  expected at : {SAMPLE_CSV}")
    print("  The gate passed on the authoring host (git rev dc595f2; notes/decisions.md D184).")
    print("  To run it here, copy game_tracking_2023_sample.csv to that path and re-run §6.")
else:
    print(f"gate sample: {SAMPLE_CSV}")
    gate_dir = RUN / "gate"
    SAMPLE_PQ = convert(SAMPLE_CSV, TRACKING_COLUMNS,
                        gate_dir / f"{SAMPLE_CSV.stem}.parquet",
                        "game_id, play_id, nfl_id, time")
    gate_receipts = detect_game_chunks(gate_dir / "parts", SAMPLE_PQ, "gate-game")
    gate_parts = sorted((gate_dir / "parts").glob("chunk_*.parquet"))
    gate_turns = pd.concat([pd.read_parquet(p) for p in gate_parts], ignore_index=True)
    if gate_turns.empty:
        raise RuntimeError("gate produced no turn rows at all — refusing to continue")
    combine_rows = detect_combine()
    if combine_rows:
        gate_turns = pd.concat([gate_turns, pd.DataFrame(combine_rows, columns=TURN_COLUMNS)],
                               ignore_index=True)
    gate_turns["valid"] = gate_turns["valid"].fillna(False).astype(bool)

    db = gate_turns[(gate_turns["setting"] == "game")
                    & (gate_turns["position_group"] == "DB") & gate_turns["valid"]]
    got = {
        "valid DB game turns": int(len(db)),
        "DB game turns, 30-60 deg (merged)": int(((db["abs_turn_angle"] >= 30)
                                                  & (db["abs_turn_angle"] < 60)).sum()),
        "DB game players": int(db["nfl_id"].nunique()),
        "DB player-games": int(db.groupby(["nfl_id", "game_id"]).ngroups),
    }
    print("\ngate numbers")
    bad = []
    for key, want in EXPECTED.items():
        ok = got[key] == want
        bad += [] if ok else [key]
        print(f"  {'PASS' if ok else 'FAIL'}  {key:<38} expected {want:>6,}  got {got[key]:>6,}")

    ref = pd.read_csv(FIG_HOLE_CSV)
    ref = ref[ref["figure"] == "fig_the_hole"]
    table = bands_table(gate_turns)
    merged = merge_30_60(table)
    diffs, compared = [], set()
    for _, row in ref.iterrows():
        cand = merged[(merged["position_group"] == "DB")
                      & (merged["setting"] == row["setting"])
                      & (merged["band"] == row["band"])]
        if cand.empty:
            diffs.append(f"{row['setting']}/{row['band']}: absent from the notebook's table")
            continue
        compared.add(row["setting"])
        g = cand.iloc[0]
        for col in ("count", "setting_total"):
            if int(g[col]) != int(row[col]):
                diffs.append(f"{row['setting']}/{row['band']}/{col}: published "
                             f"{int(row[col]):,} vs notebook {int(g[col]):,}")
        if abs(float(g["share_within_setting"]) - float(row["share_within_setting"])) > 1e-9:
            diffs.append(f"{row['setting']}/{row['band']}/share: published "
                         f"{row['share_within_setting']:.6f} vs notebook "
                         f"{g['share_within_setting']:.6f}")
    print(f"\nrow-for-row vs {FIG_HOLE_CSV.name}: {len(ref)} published rows, "
          f"{len(diffs)} difference(s); settings compared: {sorted(compared)}")
    if "combine" not in compared:
        print("  (combine rows not compared — combine files absent from this folder)")
    for d in diffs:
        print(f"  DIFF  {d}")

    print("\nDB game bands — 30–45° and 45–60° kept separate (the stored table, unmerged)")
    db_bands = table[(table["setting"] == "game") & (table["position_group"] == "DB")]
    for _, r in db_bands.iterrows():
        print(f"  {r['band']:<12} {int(r['count']):>5,}  of {int(r['setting_total']):,}  "
              f"({100 * r['share_within_setting']:5.2f}%)")
    m = merged[(merged["position_group"] == "DB") & (merged["setting"] == "game")]
    if not m.empty:
        print(f"  merged 30–60°  {int(m.iloc[0]['count']):>5,}  of "
              f"{int(m.iloc[0]['setting_total']):,}  "
              f"({100 * m.iloc[0]['share_within_setting']:5.2f}%)")

    if bad or diffs:
        print("\nGATE FAILED — stopping before any full-data work.")
        print("  Nothing was written to the full-data folders. The diff above is what needs"
              " fixing first.")
        raise RuntimeError(f"fidelity gate failed: {bad + diffs}")
    print("\n" + "-" * 76)
    print("  FIDELITY GATE PASSED — this notebook reproduces the published sample exactly")
    print("-" * 76)
    GATE_READY = True

## 7. The full run — chunked, resumable, with an ETA

Runs the engine over the file `INPUT_MODE` names. Each group of `resources.game_chunk_games`
games is written to its own checkpoint, so a closed laptop resumes at the next unfinished chunk.
Progress prints elapsed time and an ETA. The tracking file is never loaded whole.

In [ ]:
run_dir = RUN / f"run_{INPUT_MODE}"
print(f"run folder: {run_dir}\n")
t_run = time.time()
receipts = detect_game_chunks(run_dir / "parts", GAME_PQ, f"run-{INPUT_MODE}")
n_seconds = sum(float(r["seconds"]) for r in receipts)
n_frames = sum(int(r["frames"]) for r in receipts)
print(f"run complete: {time.time() - t_run:,.1f}s wall, "
      f"{n_seconds / 60:,.1f} min of chunk time, {n_frames:,} frames")
if n_frames:
    print(f"throughput: {n_frames / max(n_seconds, 1e-9):,.0f} frames/s")

## 8. Artefacts

Reads every checkpoint back (they are small), applies the reporting bands, and writes three
files into `bdb27_run/artefacts/`. Nothing outside `bdb27_run/` is touched: no existing figure,
CSV or manifest in `outputs/` changes.

In [ ]:
import hashlib
import resource

ART = RUN / "artefacts"
ART.mkdir(parents=True, exist_ok=True)

parts = sorted((run_dir / "parts").glob("chunk_*.parquet"))
if not parts:
    raise RuntimeError("no chunk checkpoints to assemble")
turns = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
if INPUT_MODE == "full":
    combine_rows = detect_combine()
    if combine_rows:
        turns = pd.concat([turns, pd.DataFrame(combine_rows, columns=TURN_COLUMNS)],
                          ignore_index=True)
turns["valid"] = turns["valid"].fillna(False).astype(bool)
if turns.empty:
    raise RuntimeError("the run produced no turn rows — nothing to write")

# ---- 1. bands: position_group x band x setting, figure 1's six reporting bands ----------
bands = merge_30_60(bands_table(turns))
bands_out = ART / "bdb27_full_game_bands.csv"
bands.to_csv(bands_out, index=False)
print(f"bands    : {bands_out.name}  {bands_out.stat().st_size:,} bytes  ({len(bands)} rows)")

# ---- 2. DB turn rows, exactly the pipeline's turn schema --------------------------------
db_turns = turns[(turns["setting"] == "game")
                 & (turns["position_group"] == "DB")][TURN_COLUMNS].reset_index(drop=True)
turns_out = ART / "bdb27_full_db_turns.parquet"
pq.write_table(arrow_table(db_turns), turns_out, compression="zstd")
print(f"DB turns : {turns_out.name}  {turns_out.stat().st_size:,} bytes  "
      f"({len(db_turns):,} rows)")


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


# ---- 3. manifest ------------------------------------------------------------------------
game = turns[turns["setting"] == "game"]
db_valid = game[(game["position_group"] == "DB") & game["valid"]]
manifest = {
    "notebook": "bdb27_full_game_turns.ipynb",
    "input_mode": INPUT_MODE,
    "input_file": GAME_CSV.name,
    "config_sha": common.config_fingerprint(cfg),
    "git_rev": common.git_rev(REPO),
    "primary_smoother": ENGINE.primary,
    "speed_floor_game": float(cfg.get("detector.speed_floor_game")),
    "chunk_games": int(cfg.get("resources.game_chunk_games")),
    "fidelity_gate": "passed" if GATE_READY else "skipped-absent",
    "counts": {
        "tracking_rows": int(n_rows), "games": int(game["game_id"].nunique()),
        "plays": int(game.groupby(["game_id", "play_id"]).ngroups),
        "players": int(game["nfl_id"].nunique()),
        "player_games": int(game.groupby(["nfl_id", "game_id"]).ngroups),
        "turn_rows": int(len(turns)),
        "valid_db_turns": int(len(db_valid)),
        "valid_db_players": int(db_valid["nfl_id"].nunique()),
        "valid_db_player_games": int(db_valid.groupby(["nfl_id", "game_id"]).ngroups),
        "valid_db_30_60": int(((db_valid["abs_turn_angle"] >= 30)
                               & (db_valid["abs_turn_angle"] < 60)).sum()),
    },
    "chunks": receipts,
    "artefacts": {
        bands_out.name: {"bytes": bands_out.stat().st_size, "sha256": sha256(bands_out)},
        turns_out.name: {"bytes": turns_out.stat().st_size, "sha256": sha256(turns_out)},
    },
    "runtime_seconds": round(time.time() - t_start, 1),
    "peak_rss_mb": round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1),
}
man_out = ART / "bdb27_full_game_manifest.json"
man_out.write_text(json.dumps(manifest, indent=2))
print(f"manifest : {man_out.name}  {man_out.stat().st_size:,} bytes")

print(f"\nfolder to upload: {ART}")
print(json.dumps(manifest["counts"], indent=2))

## 9. What to send back

Send these three files from the folder printed above (`BDB27/bdb27_run/artefacts/`):

1. `bdb27_full_game_bands.csv` — the counts by position group, band and setting.
2. `bdb27_full_db_turns.parquet` — the DB turn rows, one row per turn, the same columns as
   `outputs/turns/turns.parquet`.
3. `bdb27_full_game_manifest.json` — modes, config fingerprint, git revision, every count, the
   per-chunk timings and the SHA-256 of the other two files.

If §6 printed a red `GATE FAILED` line, **send nothing**: the notebook stopped on purpose and the
diff above is what needs fixing first.